# Topic Modeling with LDA
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/05_NLP_Embeddings/topic_modeling_lda.ipynb)

Latent Dirichlet Allocation discovers hidden THEMES in a document collection: every document is a mixture of topics, every topic a distribution over words - unsupervised, no labels needed.

We mine the classic 20 Newsgroups corpus with scikit-learn's LDA.

In [ ]:
!pip install -q scikit-learn numpy

## 1. Corpus + vectorization

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer

cats = ["sci.space", "rec.sport.baseball", "talk.politics.mideast", "comp.graphics"]
docs = fetch_20newsgroups(subset="train", categories=cats,
                          remove=("headers", "footers", "quotes")).data
print(len(docs), "documents")

cv = CountVectorizer(max_features=5000, stop_words="english",
                     min_df=5, max_df=0.4, token_pattern=r"[a-zA-Z]{3,}")
X = cv.fit_transform(docs)
print("bag-of-words:", X.shape)

LDA needs raw COUNTS (not tf-idf): its generative story is literally 'counts of words drawn from topics'.

## 2. Fit LDA with 4 topics

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

lda = LatentDirichletAllocation(n_components=4, max_iter=20,
                                learning_method="online", random_state=42)
doc_topic = lda.fit_transform(X)          # docs x topics weights

words = cv.get_feature_names_out()
for k, topic in enumerate(lda.components_):
    top = topic.argsort()[-10:][::-1]
    print(f"topic {k}:", ", ".join(words[i] for i in top))

The four themes recover the four newsgroups almost perfectly - without ever seeing labels.

## 3. Assign each document its dominant topic

In [ ]:
import numpy as np
dominant = doc_topic.argmax(axis=1)
sample = [0, 50, 500, 1500]
for i in sample:
    print(f"[topic {dominant[i]}] {docs[i][:90].strip()}...")

## 4. Choosing K: coherence-ish sweep

In [ ]:
import matplotlib.pyplot as plt
scores = {}
for k in [2, 3, 4, 5, 6, 8]:
    m = LatentDirichletAllocation(n_components=k, max_iter=10,
                                  learning_method="online", random_state=42)
    m.fit(X)
    scores[k] = m.score(X[:300])
plt.plot(list(scores), list(scores.values()), "o-")
plt.xlabel("n_topics"); plt.ylabel("log-likelihood (higher=better)")
plt.title("Elbow hunt for K"); plt.show()

## Practical guidance
| Knob | Effect |
|---|---|
| `min_df` / `max_df` | kills rare junk & boilerplate - biggest quality lever |
| `n_components` | use elbow + eyeball; there is no true K |
| short texts (tweets) | prefer NMF or BERTopic (embeddings-based) |

Gensim alternative offers `CoherenceModel` for principled K selection:
`gensim.models.LdaModel` + `CoherenceModel(model=..., texts=..., dictionary=...)`.
Modern default for production: **BERTopic** - clusters sentence-transformer embeddings instead of counting words.